### Section 3. Application Development

### Embedding Models and Vectorization

### What are Embeddings
- Vector representation of text in high-dimensional space  
- Similar meaning → vectors are closer  
### Example
- “health insurance coverage” → vector (e.g., 768-dim)  
- Similar text → close vectors  
- Unrelated text → far apart  

### What Embeddings Enable
- Search by meaning (semantic search)  
- Retrieve relevant chunks  
- Connect unstructured text with structured queries 

### Key Properties of Embeddings
- High-dimensional → typically 384–1536 dimensions  
- Dense → every value contributes (not sparse)  
- Normalized → often unit length (norm = 1)  
- Semantic similarity → closer vectors = similar meaning  
- Model-specific → each model has its own vector space  

### Cosine Similarity

- Measures similarity between two vectors  

- Formula:
  cosine_similarity(A, B) = (A · B) / (||A|| × ||B||)

- Range:
  - 1 → very similar  
  - 0 → unrelated  
  - -1 → opposite  

### Why Cosine Similarity
- Focuses on direction, not magnitude  
- Works well with normalized embeddings  

### One-line Summary
**Embeddings → vectors | Cosine similarity → find closest meaning**

### Types of Embedding Models on Databricks
### 1. databricks-bge-large-en
- Default embedding model in Databricks  
- 1024 dimensions, trained on multi-domain data  
- Best for semantic search, RAG, classification  
- Fine-tuned version of BAAI BGE  
### 2. Open Source Models (Hugging Face)
- Examples → sentence-transformers/all-MiniLM-L6-v2, mteb/e5-large  
- Can be hosted on Databricks  
- Lightweight and fast  
- May have lower performance for enterprise use cases  
### 3. Proprietary Models (OpenAI)
- Example → text-embedding-ada-002  
- Accessed via external APIs  
- Higher cost and latency  
- May have compliance/data residency concerns  
### 4. Custom Fine-Tuned Models
- Trained on domain-specific data (legal, finance, healthcare)  
- Hosted via Model Registry or Model Serving  
- Best for high-accuracy enterprise applications  
### Key Insight
- Model choice impacts:
  - retrieval quality  
  - latency  
  - cost  
  - compliance  

---

### One-line Summary
**Embedding model = foundation of RAG quality → choose based on accuracy, cost, and domain**

### Vectorizing Data on Databricks
### Step-by-Step Process
1. Extract Documents  
  - Use loaders (PDF, HTML, CSV, etc.)
2. Chunk Documents  
  - Use semantic or recursive chunking  
  - Example → 500 tokens with 100 overlap  
3. Embed Chunks  
  - Convert each chunk into vector embeddings  
4. Store in Delta Table  
  - Store text, metadata, and embedding vectors  
5. Index with Mosaic AI Vector Search  
  - Enable fast semantic retrieval  

### Storing Embeddings in Delta Lake
  - Each vector = list of float values  
  - Length = embedding model dimensions 
### Key Insight
**Delta Lake = storage + Vector Search = retrieval → together power RAG**

In [0]:
# Imports
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import DatabricksEmbeddings

# 1. Load document
loader = PyPDFLoader("/dbfs/data/policy.pdf")
docs = loader.load()

# 2. Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(docs)

# 3. Generate embeddings
embedding_model = DatabricksEmbeddings(model="databricks-bge-large-en")
vectors = embedding_model.embed_documents(
    [doc.page_content for doc in chunks]
)

# 4. Prepare data for Delta table
data = [
    {
        "chunk_id": str(i),
        "text": chunks[i].page_content,
        "vector": vectors[i],
        "doc_title": "policy.pdf",
        "tags": ["insurance"]
    }
    for i in range(len(chunks))
]

# Convert to Spark DataFrame
df = spark.createDataFrame(data)

# 5. Store in Delta Lake
df.write.format("delta") \
  .mode("append") \
  .save("/mnt/rag/embedded_chunks")

### Best Practices for Embedding Pipelines

| Practice | Why It Matters |
|----------|---------------|
| Use domain-relevant embedding models | Improves search accuracy |
| Avoid stopwords-only chunks | Reduces noise in vector space |
| Embed both queries and documents using the same model | Keeps vectors in same semantic space |
| Normalize vectors if model does not | Improves cosine similarity results |
| Store embeddings with metadata | Enables hybrid search (vector + filters) |
| Enable Change Data Feed (CDF) | Keeps embeddings updated as data changes |

---

### Key Insight
**Good embeddings = better retrieval = better RAG performance**

---

### Exam Tip
- Same embedding model for query + docs → VERY important  
- Metadata + embeddings → enables filtering + traceability  
- CDF → used for incremental re-embedding (common MCQ)

### Evaluating Embedding Model Quality

### 1. Manual Testing
- Input: User query  
- Output: Top-K similar chunks  
- Expectation: Retrieved chunks should answer the query without LLM  

---

### 2. Embedding Benchmarks

#### MTEB (Massive Text Embedding Benchmark)
- Standard benchmark to evaluate embedding models  
- Covers multiple tasks:
  - Retrieval  
  - Clustering  
  - Classification  
  - Semantic similarity  
- Used to compare models like BGE, E5, OpenAI embeddings  

#### BEIR (Benchmark for Information Retrieval)
- Focused on retrieval tasks  
- Uses real-world datasets (e.g., FAQs, finance, biomedical)  
- Measures how well embeddings retrieve relevant documents  

#### Internal Datasets
- Company-specific Q&A or documents  
- Best for domain-specific evaluation  
- More practical than public benchmarks  

---

### 3. LLM-Assisted Feedback Loop
- Use RAG + LLM to evaluate response quality  
- Identify bad answers → trace back to retrieval  
- Improve embeddings, chunking, or metadata  

---

### Key Insight
**If retrieval is bad → LLM cannot fix it → embeddings are the foundation**

---

### Exam Tips
- MTEB → general embedding benchmark (multi-task)  
- BEIR → retrieval-specific benchmark  
- Internal dataset → most important for enterprise use  
- Always evaluate embeddings BEFORE tuning LLM

### Databricks Vector Search Fundamentals
### What is Vector Search?
- Finds information by **meaning**, not exact keywords  
- Converts text into **vectors (numerical representations)**  
- Uses similarity metrics (e.g., cosine similarity) to retrieve relevant content  
### How It Works
1. Convert documents into vectors (embeddings)  
2. Store vectors in Delta Lake  
3. Convert user query into a vector (same model)  
4. Compare query vector with stored vectors  
5. Return **top-K most similar results**  
---
### Mosaic AI Vector Search
- Native Databricks vector search engine  
- Designed for scalable and enterprise use  
#### Capabilities
- Index billions of vectors from Delta Lake  
- Fast semantic top-K retrieval  
- Hybrid search (vector + metadata filters)  
- Integration with LangChain and ChatDatabricks  
- Governed via Unity Catalog  
- Supports Change Data Feed (real-time updates)  
---
### Architecture Components
- Delta Tables → Store embeddings and metadata  
- Model Serving → Generate embeddings  
- Vector Search Endpoint → Perform similarity search  
- Unity Catalog → Access control and governance  
- Serverless Compute → Runs search infrastructure  
---
### Key Insight
**Vector Search = Embeddings + Similarity Search + Metadata Filtering**
### Exam Tips
- Query + documents must use SAME embedding model  
- Cosine similarity → most commonly used metric  
- Delta Lake → storage layer  
- Vector Search → retrieval layer  
- Unity Catalog → governance  

---

### One-line Summary
**Vector search enables semantic retrieval by comparing embeddings instead of keywords**

### Step-by-Step: Creating a Vector Search Index
### Step 1: Store Embeddings in Delta Table
- Store chunks, embeddings, metadata
- Fields: chunk_id(STRING),
- text(STRING), 
- vector(ARRAY<FLOAT>), 
- doc_title(STRING), 
- tags(ARRAY<STRING>), 
- last_updated(TIMESTAMP)
### Step 2: Register Table in Unity Catalog

In [0]:
%sql
CREATE TABLE rag_catalog.rag_schema.policy_chunks USING DELTA LOCATION '/mnt/vector_data/policy_chunks';

###Step 3: Enable Change Data Feed (CDF)

In [0]:
%sql
ALTER TABLE rag_catalog.rag_schema.policy_chunks
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

###Step 4: Create Vector Search Index

In [0]:
%sql
CREATE VECTOR INDEX idx_policy_chunks ON TABLE rag_catalog.rag_schema.policy_chunks 
EMBEDDING_COLUMN vector 
TEXT_COLUMN text 
METADATA_COLUMNS (doc_title,tags);

###Step 5: Query the Vector Index

In [0]:
%sql
SELECT text FROM VECTOR_SEARCH(
  INDEX idx_policy_chunks,
  QUERY 'Explain eligibility for maternity leave',
  TOP_K 5,
  FILTER 'tags IN ("HR","Employee Benefits")');

###Sample Output
- Returns top-K similar chunks with score
- Supports metadata filtering

- Key Insight
  - Vector Search=Semantic similarity+Metadata filtering
- Exam Tips
  - CDF required for updates
  - Vector must be ARRAY<FLOAT>
  - Unity Catalog required  
  - Same embedding model for query+docs

### Integration with LangChain (Python)
- LangChain integrates Mosaic AI Vector Search into GenAI pipelines
### Steps
1. Connect to Vector Search index
2. Connect to LLM (ChatDatabricks)
3. Create RetrievalQA chain
4. Query the system
### Code Example


In [0]:
%python
from langchain.vectorstores import DatabricksVectorSearch
from langchain.chat_models import ChatDatabricks
from langchain.chains import RetrievalQA

vectorstore=DatabricksVectorSearch(index_name="idx_policy_chunks")

llm=ChatDatabricks(endpoint="databricks-meta-llama-3-1-70b-instruct")

rag_chain=RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True)
    
query="What are the exclusions in life insurance coverage?"
print(rag_chain.run(query))

### Integrating Vector Search into GenAI Pipelines
### Why Integration Matters
- User asks question
- Vector search retrieves context
- Prompt is constructed dynamically
- LLM generates grounded answer
- Output is explainable and production-ready
### RAG Architecture Recap
User Query→Embed Query→Vector Search (Top-K)→Construct Prompt→LLM Generate→Return Answer+Sources
Layers:
- Retrieval→Mosaic AI Vector Search
- Chaining→LangChain
- Generation→ChatDatabricks
- Governance→Unity Catalog